#### 7. Extend the pipeline to handle a 'dirty data' scenario not covered in class (e.g., mixed date formats, currency symbols in numeric columns) and document your approach.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DateType
import re

orders_df = spark.table("samples.tpch.orders")
customer_df = spark.table("samples.tpch.customer")

dirty_orders = orders_df.limit(1000).withColumn(
    "dirty_order_date",
    F.when(F.rand() < 0.3, F.date_format(F.col("o_orderdate"), "MM/dd/yyyy"))
     .when(F.rand() < 0.6, F.date_format(F.col("o_orderdate"), "yyyy-MM-dd"))
     .otherwise(F.date_format(F.col("o_orderdate"), "dd-MMM-yyyy"))
)

dirty_orders = dirty_orders.withColumn(
    "dirty_totalprice",
    F.when(F.rand() < 0.4, F.concat(F.lit("$"), F.col("o_totalprice")))
     .when(F.rand() < 0.7, F.concat(F.lit("USD "), F.col("o_totalprice")))
     .otherwise(F.concat(F.col("o_totalprice"), F.lit(" USD")))
)

dirty_orders.select("o_orderdate", "dirty_order_date", "o_totalprice", "dirty_totalprice").show(10, truncate=False)

# DATA CLEANING PIPELINE
def clean_mixed_dates(date_col):
    return F.coalesce(
        F.try_to_date(date_col, "yyyy-MM-dd"),      # ISO format
        F.try_to_date(date_col, "MM/dd/yyyy"),      # US format
        F.try_to_date(date_col, "dd-MMM-yyyy")      # European format
    )

def clean_currency(price_col):
    # Remove currency symbols and text, keep only digits and decimal point
    cleaned = F.regexp_replace(price_col, r"[^0-9.]", "")
    return cleaned.cast(DecimalType(15, 2))

# Apply cleaning transformations
cleaned_orders = dirty_orders.withColumn(
    "cleaned_order_date",
    clean_mixed_dates(F.col("dirty_order_date"))
).withColumn(
    "cleaned_totalprice",
    clean_currency(F.col("dirty_totalprice"))
)

print("\n" + "="*80)
print("CLEANED DATA RESULTS")
print("="*80)

result = cleaned_orders.select(
    "dirty_order_date",
    "cleaned_order_date",
    "dirty_totalprice",
    "cleaned_totalprice",
    "o_totalprice"
)
result.show(20, truncate=False)



#### 8. Set up a branching strategy (dev/main) for the Cyntexa analytics repo and write a short guide for teammates on the pull-request review workflow before merging into main.

We follow a simple two-branch model that keeps things clean and predictable:

### **`main` branch**
- This is our production-ready code
- Always stable, always deployable
- Protected – no one pushes directly to main
- All changes come through reviewed pull requests

### **`dev` branch**
- Our integration branch where features come together
- More relaxed than main, but still tested
- Acts as a staging area before production
- Regular merges to main after validation

### **Feature branches**
- Created from `dev` for each new piece of work
- Naming convention: `feature/your-feature-name` or `fix/bug-description`
- Examples: `feature/customer-segmentation`, `fix/date-parsing-error`
- Deleted after merging to keep the repo tidy

---

## Pull Request Workflow – A Teammate's Guide

### Step 1: Starting Your Work
```bash
# Make sure your dev branch is up to date
git checkout dev
git pull origin dev

# Create your feature branch
git checkout -b feature/my-awesome-analysis
```

### Step 2: Doing the Work
- Write your code, run your notebooks, test your queries
- Commit regularly with clear messages:
  ```bash
  git commit -m "Add customer churn prediction model"
  git commit -m "Fix currency formatting in revenue reports"
  ```
- Push your branch to remote:
  ```bash
  git push origin feature/my-awesome-analysis
  ```

### Step 3: Opening a Pull Request

1. **Go to GitHub/GitLab** and create a PR from your feature branch → `dev`
2. **Write a helpful description:**
   - What does this PR do?
   - Why are we making this change?
   - Any context reviewers should know?
   - Screenshots or results if relevant
3. **Request reviewers** – tag at least one teammate (two for critical changes)
4. **Link any related issues** or tickets

### Step 4: The Review Process (For Reviewers)

When you're asked to review, you're helping keep our codebase healthy:
1. Read all comments thoughtfully
2. Make requested changes and push new commits
3. Reply to comments explaining what you changed
4. Mark conversations as resolved once addressed
5. Don't take feedback personally – it's about the code, not you!

### Step 6: Merging to `dev`

Once approved:
1. **Squash and merge** (or merge commit, depending on team preference)
2. Delete your feature branch
3. Done! 

### Step 7: Promoting to `main`

Periodically (weekly or after major milestones):
1. Create a PR from `dev` → `main`
2. Final review by a senior team member
3. Ensure all tests pass
4. Merge to main
5. Tag the release if appropriate: `v1.0.0`


#### 9. (Data Analyst) Using the cleaned dataset, produce a summary report answering 3 business questions (e.g., top-selling category per region, month-over-month growth, average order value trend) and note any data-quality caveats a stakeholder should know about.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cleaned_orders = cleaned_orders
customer_df = customer_df

lineitem_df = spark.table("samples.tpch.lineitem")
part_df = spark.table("samples.tpch.part")
region_df = spark.table("samples.tpch.region")
nation_df = spark.table("samples.tpch.nation")

In [0]:

regional_sales = cleaned_orders \
    .join(lineitem_df, cleaned_orders.o_orderkey == lineitem_df.l_orderkey) \
    .join(part_df, lineitem_df.l_partkey == part_df.p_partkey) \
    .join(customer_df, cleaned_orders.o_custkey == customer_df.c_custkey) \
    .join(nation_df, customer_df.c_nationkey == nation_df.n_nationkey) \
    .join(region_df, nation_df.n_regionkey == region_df.r_regionkey)

# Aggregate sales by region and product type
region_product_sales = regional_sales \
    .groupBy("r_name", "p_type") \
    .agg(
        F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))).alias("total_revenue"),
        F.sum("l_quantity").alias("total_quantity")
    )

# Rank product types within each region
window_spec = Window.partitionBy("r_name").orderBy(F.desc("total_revenue"))
top_products_by_region = region_product_sales \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("r_name", "rank")

print("\nTop 3 Product Types by Revenue per Region:")
top_products_by_region.show(15, truncate=False)

In [0]:
# Use cleaned_order_date from previous cleaning
monthly_metrics = cleaned_orders \
    .withColumn("order_month", F.date_trunc("month", F.col("cleaned_order_date"))) \
    .groupBy("order_month") \
    .agg(
        F.count("o_orderkey").alias("order_count"),
        F.sum("cleaned_totalprice").alias("total_revenue")
    ) \
    .orderBy("order_month")

# Calculate month-over-month growth
window_lag = Window.orderBy("order_month")
mom_growth = monthly_metrics \
    .withColumn("prev_month_orders", F.lag("order_count").over(window_lag)) \
    .withColumn("prev_month_revenue", F.lag("total_revenue").over(window_lag)) \
    .withColumn(
        "order_growth_pct",
        ((F.col("order_count") - F.col("prev_month_orders")) / F.col("prev_month_orders") * 100)
    ) \
    .withColumn(
        "revenue_growth_pct",
        ((F.col("total_revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue") * 100)
    ) \
    .select(
        "order_month",
        "order_count",
        "total_revenue",
        F.round("order_growth_pct", 2).alias("order_growth_%"),
        F.round("revenue_growth_pct", 2).alias("revenue_growth_%")
    )

print("\nMonth-over-Month Growth Trends:")
mom_growth.show(12, truncate=False)

In [0]:
# Join orders with customers and calculate AOV by market segment over time
customer_orders = cleaned_orders \
    .join(customer_df, cleaned_orders.o_custkey == customer_df.c_custkey) \
    .withColumn("order_quarter", F.date_trunc("quarter", F.col("cleaned_order_date")))

aov_by_segment = customer_orders \
    .groupBy("order_quarter", "c_mktsegment") \
    .agg(
        F.avg("cleaned_totalprice").alias("avg_order_value"),
        F.count("o_orderkey").alias("order_count"),
        F.sum("cleaned_totalprice").alias("total_revenue")
    ) \
    .withColumn("avg_order_value", F.round("avg_order_value", 2)) \
    .orderBy("order_quarter", "c_mktsegment")

print("\nAverage Order Value Trend by Market Segment (Quarterly):")
aov_by_segment.show(20, truncate=False)

# Pivot for easier comparison
aov_pivot = aov_by_segment \
    .groupBy("order_quarter") \
    .pivot("c_mktsegment") \
    .agg(F.first("avg_order_value")) \
    .orderBy("order_quarter")

print("\nAverage Order Value by Segment (Pivot View):")
aov_pivot.show(truncate=False)